In [1]:
!pip install TTS

  Using cached tts-0.22.0-cp310-cp310-macosx_15_0_arm64.whl
  Using cached numpy-1.22.0-cp310-cp310-macosx_11_0_arm64.whl.metadata (2.0 kB)
  Using cached inflect-7.5.0-py3-none-any.whl.metadata (24 kB)
  Using cached anyascii-0.3.3-py3-none-any.whl.metadata (1.6 kB)
  Using cached pysbd-0.3.4-py3-none-any.whl.metadata (6.1 kB)
  Using cached pandas-1.5.3-cp310-cp310-macosx_11_0_arm64.whl.metadata (11 kB)
  Using cached trainer-0.0.36-py3-none-any.whl.metadata (8.1 kB)
  Using cached coqpit-0.0.17-py3-none-any.whl.metadata (11 kB)
  Using cached jieba-0.42.1-py3-none-any.whl
  Using cached pypinyin-0.55.0-py2.py3-none-any.whl.metadata (12 kB)
  Using cached hangul_romanize-0.1.0-py3-none-any.whl.metadata (1.2 kB)
  Using cached gruut-2.2.3-py3-none-any.whl
  Using cached jamo-0.4.1-py3-none-any.whl.metadata (2.3 kB)
  Using cached g2pkk-0.1.2-py3-none-any.whl.metadata (2.0 kB)
  Using cached bangla-0.0.5-py3-none-any.whl.metadata (4.7 kB)
  Using cached bnnumerizer-0.0.2-py3-none-any.w

In [1]:
import torch

# Monkey-patch torch.load to always use weights_only=False
# Use this ONLY if you trust the source (Coqui XTTS is safe)
original_load = torch.load
torch.load = lambda *args, **kwargs: original_load(*args, **{**kwargs, 'weights_only': False})

from TTS.api import TTS
xtts_model = TTS('tts_models/multilingual/multi-dataset/xtts_v2')


 > tts_models/multilingual/multi-dataset/xtts_v2 is already downloaded.


/Users/robbannn/.pyenv/versions/3.10.12/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


 > Using model: xtts


GPT2InferenceModel has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.


In [2]:
from TTS.api import TTS

# This will return the local paths of the model files (it won't redownload if they already exist)
model_path, config_path, vocoder_path, vocoder_config_path, model_dir = TTS().download_model_by_name("tts_models/multilingual/multi-dataset/xtts_v2")

print(f"Model Directory: {model_dir}")


 > tts_models/multilingual/multi-dataset/xtts_v2 is already downloaded.
Model Directory: /Users/robbannn/Library/Application Support/tts/tts_models--multilingual--multi-dataset--xtts_v2


In [3]:
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import Xtts

config = XttsConfig()
# Use the absolute path you already used for the config
model_path = "/Users/robbannn/Desktop/PROJECTS/urduTTS/models/xtts_v2/"

config.load_json(model_path + "config.json")

model = Xtts.init_from_config(config)

# Pass the directory path, NOT the model.pth file
model.load_checkpoint(config, checkpoint_dir=model_path, eval=True)


In [4]:
from trainer import Trainer, TrainerArgs
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.datasets import load_tts_samples
from TTS.tts.models.xtts import Xtts

#  PATHS 
XTTS_CHECKPOINT = "/Users/robbannn/Desktop/PROJECTS/urduTTS/models/xtts_v2/"
DATASET_PATH    = "/Users/robbannn/Desktop/PROJECTS/urduTTS/xtts_dataset/"
OUTPUT_PATH     = "/Users/robbannn/Desktop/PROJECTS/urduTTS/xtts_dataset/output/"

# CONFIG 
config = XttsConfig()
config.load_json(XTTS_CHECKPOINT + "config.json")

config.output_path = OUTPUT_PATH
config.epochs      = 10
config.batch_size  = 4
config.eval_batch_size = 2
config.lr          = 5e-6

In [5]:
import os
import sys
import TTS.tts.datasets as datasets
from TTS.tts.configs.shared_configs import BaseDatasetConfig

def urdu_formatter(root_path, meta_file, **kwargs):
    """
    Custom formatter for Urdu dataset (ID|Text format).
    """
    txt_file = os.path.join(root_path, meta_file)
    items = []
    folder = os.path.dirname(meta_file) 
    
    with open(txt_file, "r", encoding="utf-8") as f:
        for line in f:
            cols = line.strip().split("|")
            if len(cols) >= 2:
                wav_file = os.path.join(root_path, folder, "wavs", cols[0] + ".wav")
                text = cols[1]
                items.append({
                    "text": text,
                    "audio_file": wav_file,
                    "speaker_name": "urdu_speaker",
                    "root_path": root_path,
                    "language": "ur"
                })
    return items



In [6]:
# Register the formatter
datasets.urdu_formatter = urdu_formatter

# Define the dataset configuration
dataset_config = BaseDatasetConfig(
    formatter="urdu_formatter",
    dataset_name="urdu_tts",
    path=DATASET_PATH,
    meta_file_train="train/metadata.csv",
    meta_file_val="val/metadata.csv",
)

In [7]:


# DATASET 
# Set eval_split=True so the library processes the meta_file_val
train_samples, eval_samples = load_tts_samples(
    datasets=[dataset_config],
    eval_split=True, 
)

# Robustness check to prevent NoneType errors in print
train_samples = train_samples or []
# Optional: Separate a small test set that the trainer never sees
test_samples = eval_samples[:100]  # Keep first 100 for your own testing later
eval_samples = eval_samples[100:]  # Use the rest for the Trainer's validation


print(f" > Loaded {len(train_samples)} training samples and {len(eval_samples)} evaluation samples.")


 | > Found 7538 files in /Users/robbannn/Desktop/PROJECTS/urduTTS/xtts_dataset
 > Loaded 7538 training samples and 842 evaluation samples.


In [8]:
# MODEL 
model = Xtts.init_from_config(config)
model.load_checkpoint(config, checkpoint_dir=XTTS_CHECKPOINT, eval=False)

In [ ]:
from TTS.utils.audio import AudioProcessor

# FREEZE 
for name, param in model.named_parameters():
    param.requires_grad = False

for name, param in model.named_parameters():
    if "gpt" in name or "text_embedding" in name:
        param.requires_grad = True

# --- BLANKET COMPATIBILITY PATCHES ---

# UPDATED: Initialize AudioProcessor with safe defaults to avoid the TypeError
if not hasattr(model, "ap") or model.ap is None:
    print(" > Initializing AudioProcessor with safe defaults")
    # We provide standard values so the internal calculations don't fail
    safe_audio_config = {
        "sample_rate": config.audio.get("sample_rate", 22050),
        "resample": True,
        "num_mels": 80,
        "fft_size": 1024,
        "hop_length": 256,
        "win_length": 1024,
    }
    model.ap = AudioProcessor(**safe_audio_config)

# 1. Patch Tokenizer
if hasattr(model, "tokenizer") and model.tokenizer is not None:
    model.tokenizer.use_phonemes = False
    model.tokenizer.print_logs = lambda *args, **kwargs: None 

# 2. Patch Model Criterion
model.get_criterion = lambda: None

# 3. Patch Speaker & Language Managers
for manager_name in ["speaker_manager", "language_manager"]:
    manager = getattr(model, manager_name, None)
    if manager is not None:
        alt_method = getattr(manager, f"save_{manager_name.split('_')[0]}_ids", None)
        manager.save_ids_to_file = alt_method if alt_method else lambda *args, **kwargs: None

# 4. Blanket Patch for Model Args
if hasattr(model, "args"):
    common_flags = [
        "use_speaker_embedding", "use_language_embedding", "use_d_vector_file",
        "d_vector_dim", "use_speaker_encoder", "speaker_encoder_model_path",
        "speaker_encoder_config_path", "model_args"
    ]
    for flag in common_flags:
        if not hasattr(model.args, flag):
            val = False if flag.startswith("use_") else None
            setattr(model.args, flag, val)

# --- START TRAINING ---

trainer = Trainer(
    TrainerArgs(restore_path=None, skip_train_epoch=False),
    config,
    output_path=OUTPUT_PATH,
    model=model,
    train_samples=train_samples,
    eval_samples=eval_samples,
)

trainer.fit()


 > Training Environment:
 | > Backend: Torch
 | > Mixed precision: False
 | > Precision: float32
 | > Num. of CPUs: 8
 | > Num. of Torch Threads: 4
 | > Torch seed: 54321
 | > Torch CUDNN: True
 | > Torch CUDNN deterministic: False
 | > Torch CUDNN benchmark: False
 | > Torch TF32 MatMul: False
 > Start Tensorboard: tensorboard --logdir=/Users/robbannn/Desktop/PROJECTS/urduTTS/xtts_dataset/output/run-May-10-2026_04+06PM-d372552

 > Model has 466874863 parameters


 > Setting up Audio Processor...
 | > sample_rate:22050
 | > resample:True
 | > num_mels:80
 | > log_func:np.log10
 | > min_level_db:0
 | > frame_shift_ms:None
 | > frame_length_ms:None
 | > ref_level_db:None
 | > fft_size:1024
 | > power:None
 | > preemphasis:0.0
 | > griffin_lim_iters:None
 | > signal_norm:None
 | > symmetric_norm:None
 | > mel_fmin:0
 | > mel_fmax:None
 | > pitch_fmin:None
 | > pitch_fmax:None
 | > spec_gain:20.0
 | > stft_pad_mode:reflect
 | > max_norm:1.0
 | > clip_norm:True
 | > do_trim_silence:False
 | > trim_db:60
 | > do_sound_norm:False
 | > do_amp_to_db_linear:True
 | > do_amp_to_db_mel:True
 | > do_rms_norm:False
 | > db_level:None
 | > stats_path:None
 | > base:10
 | > hop_length:256
 | > win_length:1024
 > `speakers.pth` is saved to /Users/robbannn/Desktop/PROJECTS/urduTTS/xtts_dataset/output/run-May-10-2026_04+06PM-d372552/speakers.pth.
 > `speakers_file` is updated in the config.json.
 > `language_ids.json` is saved to /Users/robbannn/Desktop/PROJECTS/u


 > EPOCH: 0/10
 --> /Users/robbannn/Desktop/PROJECTS/urduTTS/xtts_dataset/output/run-May-10-2026_04+06PM-d372552




> DataLoader initialization
| > Tokenizer:
| > Number of instances : 7538



 > TRAINING (2026-05-10 16:06:20) 
 ! Run is removed from /Users/robbannn/Desktop/PROJECTS/urduTTS/xtts_dataset/output/run-May-10-2026_04+06PM-d372552


 | > Preprocessing samples
 | > Max text length: 124
 | > Min text length: 2
 | > Avg text length: 36.21743167949058
 | 
 | > Max audio length: 230996
 | > Min audio length: 22227
 | > Avg audio length: 91725.60493499602
 | > Num. instances discarded samples: 0
 | > Batch group size: 0.


Traceback (most recent call last):
  File "/Users/robbannn/.pyenv/versions/3.10.12/lib/python3.10/site-packages/trainer/trainer.py", line 1833, in fit
    self._fit()
  File "/Users/robbannn/.pyenv/versions/3.10.12/lib/python3.10/site-packages/trainer/trainer.py", line 1785, in _fit
    self.train_epoch()
  File "/Users/robbannn/.pyenv/versions/3.10.12/lib/python3.10/site-packages/trainer/trainer.py", line 1503, in train_epoch
    for cur_step, batch in enumerate(self.train_loader):
  File "/Users/robbannn/.pyenv/versions/3.10.12/lib/python3.10/site-packages/torch/utils/data/dataloader.py", line 708, in __next__
    data = self._next_data()
  File "/Users/robbannn/.pyenv/versions/3.10.12/lib/python3.10/site-packages/torch/utils/data/dataloader.py", line 764, in _next_data
    data = self._dataset_fetcher.fetch(index)  # may raise StopIteration
  File "/Users/robbannn/.pyenv/versions/3.10.12/lib/python3.10/site-packages/torch/utils/data/_utils/fetch.py", line 52, in fetch
    data = [se

SystemExit: 1

/Users/robbannn/.pyenv/versions/3.10.12/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [31]:
!pip install git+https://github.com/coqui-ai/TTS.git

  Cloning https://github.com/coqui-ai/TTS.git to /private/var/folders/0x/rx6z5n5d6dzbfbmmrfbxwsdw0000gn/T/pip-req-build-jkx230jx
  Running command git clone --filter=blob:none --quiet https://github.com/coqui-ai/TTS.git /private/var/folders/0x/rx6z5n5d6dzbfbmmrfbxwsdw0000gn/T/pip-req-build-jkx230jx
  Resolved https://github.com/coqui-ai/TTS.git to commit dbf1a08a0d4e47fdad6172e433eeb34bc6b13b4e
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached numpy-1.22.0-cp310-cp310-macosx_11_0_arm64.whl.metadata (2.0 kB)
  Using cached pandas-1.5.3-cp310-cp310-macosx_11_0_arm64.whl.metadata (11 kB)
  Using cached trainer-0.0.36-py3-none-any.whl.metadata (8.1 kB)
  Using cached coqpit-0.0.17-py3-none-any.whl.metadata (11 kB)
INFO: pip is looking at multiple versions of scipy to determine which version is compatible with other requirements. This could take a while.
  Using cached scipy-1.15.2-cp310-cp31

In [ ]:
import torch
import torch.nn as nn
from trainer import Trainer, TrainerArgs
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.datasets import load_tts_samples
from TTS.tts.models.xtts import Xtts


In [ ]:
# CUSTOM ADAPTER 
class UrduAdapter(nn.Module):
    """
    Bottleneck Adapter for Urdu language adaptation.
    Sits between Text Embedding and GPT-2.
    """
    def __init__(self, input_dim=512, bottleneck_dim=128):
        super().__init__()

        self.adapter = nn.Sequential(
            nn.Linear(input_dim, bottleneck_dim),   # compress
            nn.LayerNorm(bottleneck_dim),            # normalize
            nn.ReLU(),                               # activate
            nn.Linear(bottleneck_dim, input_dim),   # expand back
        )

        # residual scale — learnable, starts small
        self.scale = nn.Parameter(torch.ones(1) * 0.1)

    def forward(self, x):
        return x + self.scale * self.adapter(x)    # residual connection

In [ ]:

# XTTS +  ADAPTER COMBINED 
class XttsWithUrduAdapter(nn.Module):
    """
    Wraps XTTS and injects the UrduAdapter into the forward pass.
    """
    def __init__(self, xtts_model, adapter):
        super().__init__()
        self.xtts    = xtts_model
        self.adapter = adapter

    def forward(self, text_inputs, *args, **kwargs):
        # Step 1 → get text embeddings from XTTS
        embeddings = self.xtts.text_encoder(text_inputs)

        # Step 2 → pass through YOUR adapter
        adapted_embeddings = self.adapter(embeddings)

        # Step 3 → rest of XTTS runs normally
        return self.xtts.gpt(adapted_embeddings, *args, **kwargs)

In [ ]:


# PATHS 
XTTS_CHECKPOINT = "path/to/xtts_v2/"
DATASET_PATH    = "xtts_dataset/"
OUTPUT_PATH     = "xtts_dataset/output/"


# LOAD BASE XTTS 
config = XttsConfig()
config.load_json(XTTS_CHECKPOINT + "config.json")

config.output_path     = OUTPUT_PATH
config.epochs          = 20
config.batch_size      = 4
config.eval_batch_size = 2
config.lr              = 5e-6

base_model = Xtts.init_from_config(config)
base_model.load_checkpoint(
    config,
    checkpoint_dir=XTTS_CHECKPOINT,
    eval=False
)

In [ ]:

# FREEZE / UNFREEZE STRATEGY 
# EXP1 => freeze everything in base XTTS
for param in base_model.parameters():
    param.requires_grad = False

# EXP => selectively unfreeze for experimentation
for name, param in base_model.named_parameters():
    if "gpt" in name:            # unfreeze GPT-2 top layers
        param.requires_grad = True
    if "text_embedding" in name: # unfreeze text embeddings
        param.requires_grad = True

In [ ]:
# ADAPTER 
# Adapter is always trainable 
adapter = UrduAdapter(input_dim=512, bottleneck_dim=128)

In [ ]:
# combining both
model = XttsWithUrduAdapter(xtts_model=base_model, adapter=adapter)

In [ ]:
# DATASET 
train_samples, eval_samples = load_tts_samples(
    datasets=[{
        "formatter"       : "ljspeech",
        "dataset_name"    : "urdu_tts",
        "path"            : DATASET_PATH,
        "meta_file_train" : "train/metadata.csv",
        "meta_file_val"   : "val/metadata.csv",
        "language"        : "ur",
        "ignored_speakers": None,
    }],
    eval_split=True,
)

In [ ]:
# TRAIN 
trainer = Trainer(
    TrainerArgs(restore_path=None, skip_train_epoch=False),
    config,
    output_path=OUTPUT_PATH,
    model=model,
    train_samples=train_samples,
    eval_samples=eval_samples,
)

In [ ]:
trainer.fit()